# YOLO12-Small + ECA-Net — CH-RDD2022 (Kaggle)

Notebook ini melatih YOLO12-Small dengan Efficient Channel Attention (ECA-Net) pada dataset CH-RDD2022. Hasil training, checkpoint, konfigurasi, metrik, dan prediksi test dikemas ke ZIP di `/kaggle/working`.

Hyperparameter mengikuti Tabel 3 dari *BL-YOLOv8: An Improved Road Defect Detection Model Based on YOLOv8*: SGD, learning rate 0.01, momentum 0.937, weight decay 0.0005, batch 64, image size 640, dan 160 epoch.

Sebelum menjalankan, pilih **Accelerator: GPU** pada Kaggle. Aktifkan Internet bila `ultralytics` belum tersedia di environment Kaggle.

In [ ]:
# 1. Install dan cek environment Kaggle
!pip install -q -U ultralytics

import sys
import json
import platform
import zipfile
from pathlib import Path

import torch
import ultralytics

def log_section(title: str) -> None:
    line = '=' * 90
    print(f'\n{line}\n{title}\n{line}')

log_section('KAGGLE ENVIRONMENT')
print(f'Python       : {platform.python_version()}')
print(f'PyTorch      : {torch.__version__}')
print(f'Ultralytics  : {ultralytics.__version__}')
print(f'CUDA ready   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    DEVICE = 0
else:
    print('WARNING: GPU tidak ditemukan. Training akan sangat lambat pada CPU.')
    DEVICE = 'cpu'


In [ ]:
# 2. Konfigurasi dataset dan parameter training
WORKDIR = Path('/kaggle/working')
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = WORKDIR / 'yolo12s_eca.yaml'
ECA_MODULE = WORKDIR / 'eca_yolo12.py'
RUNS_DIR = WORKDIR / 'runs'

# Hyperparameter Tabel 3 paper BL-YOLOv8.
EPOCHS = 160
IMGSZ = 640
BATCH = 64
LR0 = 0.01
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
OPTIMIZER = 'SGD'
PATIENCE = 0  # Menonaktifkan early stopping agar seluruh 160 epoch dijalankan.
WORKERS = 2
SEED = 42
EXPERIMENT_NAME = 'yolo12s_eca_ch_rdd2022'

print('Paper setting: SGD | lr0=0.01 | momentum=0.937 | weight_decay=0.0005 | batch=64 | imgsz=640 | epochs=160')
print('Catatan: batch 64 digunakan paper pada RTX 3090 24 GB. Jika GPU Kaggle kehabisan memori, turunkan batch; hasilnya tidak lagi identik dengan setup paper.')

DATA_YAML.write_text(
    f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''',
    encoding='utf-8',
)

log_section('DATASET CONFIGURATION')
print(DATA_YAML.read_text())
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'

image_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / split / 'images'
    label_dir = DATA_ROOT / split / 'labels'
    image_count = sum(path.suffix.lower() in image_suffixes for path in image_dir.rglob('*')) if image_dir.exists() else 0
    label_count = len(list(label_dir.glob('*.txt'))) if label_dir.exists() else 0
    print(f'{split:>5}: {image_count:>6} images | {label_count:>6} label files | {image_dir}')
    if split in {'train', 'val'}:
        assert image_count > 0, f'Tidak ada gambar pada {image_dir}'


In [ ]:
# 3. Tambahkan modul ECA-Net dan registrasikan ke parser model Ultralytics
# File ini ikut dimasukkan ke ZIP agar checkpoint dapat dimuat kembali di notebook lain.
ECA_MODULE.write_text(
    '''import math

import torch
from torch import nn


class ECAAttention(nn.Module):
    """Efficient Channel Attention with a channel-adaptive odd Conv1D kernel."""

    def __init__(self, c1: int, gamma: int = 2, b: int = 1) -> None:
        super().__init__()
        t = int(abs((math.log2(c1) + b) / gamma))
        k = max(t if t % 2 else t + 1, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
        self.act = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.avg_pool(x).squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * self.act(y)
''',
    encoding='utf-8',
)

sys.path.insert(0, str(WORKDIR))
from eca_yolo12 import ECAAttention
import ultralytics.nn.tasks as tasks

tasks.ECAAttention = ECAAttention  # Agar nama ECAAttention dapat dibaca dari YAML.
print(f'ECAAttention registered: {tasks.ECAAttention.__module__}.{tasks.ECAAttention.__name__}')


In [ ]:
# 4. Konfigurasi YOLO12-Small + ECA. ECA diletakkan setelah backbone P3, P4, dan P5.
MODEL_YAML.write_text(
    '''# YOLO12-Small with Efficient Channel Attention
nc: 5
scales:
  s: [0.50, 0.50, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, ECAAttention, []]  # P3: 256 channels, k=5
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 4, A2C2f, [512, True, 4]]
  - [-1, 1, ECAAttention, []]  # P4: 256 channels, k=5
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 4, A2C2f, [1024, True, 1]]
  - [-1, 1, ECAAttention, []]  # P5: 512 channels, k=5

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 2, A2C2f, [512, False, -1]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 2, A2C2f, [256, False, -1]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 14], 1, Concat, [1]]
  - [-1, 2, A2C2f, [512, False, -1]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 11], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]
  - [[17, 20, 23], 1, Detect, [nc]]
''',
    encoding='utf-8',
)

log_section('YOLO12S + ECA MODEL YAML')
print(MODEL_YAML.read_text())


In [ ]:
# 5. Cetak perbandingan model standar vs. model ECA agar log Kaggle ringkas dan mudah dibaca
from ultralytics.nn.tasks import DetectionModel

baseline_model = DetectionModel('yolo12s.yaml', nc=5, verbose=False)
eca_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=False)
baseline_params = sum(parameter.numel() for parameter in baseline_model.parameters())
eca_params = sum(parameter.numel() for parameter in eca_model.parameters())
eca_layers = [
    (layer.i, layer.conv.kernel_size[0], layer.conv.padding[0])
    for layer in eca_model.model
    if isinstance(layer, ECAAttention)
]

log_section('MODEL COMPARISON')
print(f'YOLO12-Small parameters      : {baseline_params:,}')
print(f'YOLO12-Small + ECA parameters: {eca_params:,}')
print(f'Additional parameters         : {eca_params - baseline_params:,}')
print(f'ECA layers (index, kernel, pad): {eca_layers}')
print('Expected ECA layout           : P3 -> ECA(k=5), P4 -> ECA(k=5), P5 -> ECA(k=5)')

# Lepaskan model perbandingan dari memori GPU sebelum training.
del baseline_model, eca_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 6. Training. Ultralytics akan menampilkan metrik per epoch pada log Kaggle dan menyimpan grafik otomatis.
from ultralytics import YOLO

log_section('TRAINING STARTED')
print(f'Experiment : {EXPERIMENT_NAME}')
print(f'Epochs     : {EPOCHS}')
print(f'Image size : {IMGSZ}')
print(f'Batch      : {BATCH}')
print(f'Optimizer  : {OPTIMIZER}')
print(f'LR0        : {LR0}')
print(f'Momentum   : {MOMENTUM}')
print(f'Weight decay: {WEIGHT_DECAY}')
print(f'Early stop : disabled (patience={PATIENCE})')
print(f'Device     : {DEVICE}')

model = YOLO(str(MODEL_YAML))
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=EXPERIMENT_NAME,
    exist_ok=True,
    pretrained=False,
    optimizer=OPTIMIZER,
    lr0=LR0,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    cos_lr=False,
    patience=PATIENCE,
    seed=SEED,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = Path(model.trainer.best)
LAST_PT = Path(model.trainer.last)
print(f'\nRun directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')
print(f'Last weights : {LAST_PT}')


In [ ]:
# 7. Validasi best checkpoint dan evaluasi/prediksi test
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=IMGSZ,
    batch=16,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=f'{EXPERIMENT_NAME}_val',
    exist_ok=True,
    plots=True,
)
print(f'Validation mAP50-95: {val_metrics.box.map:.4f}')
print(f'Validation mAP50   : {val_metrics.box.map50:.4f}')

test_label_dir = DATA_ROOT / 'test' / 'labels'
test_has_labels = test_label_dir.exists() and any(test_label_dir.glob('*.txt'))
if test_has_labels:
    test_metrics = best_model.val(
        data=str(DATA_YAML),
        split='test',
        imgsz=IMGSZ,
        batch=16,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_test',
        exist_ok=True,
        plots=True,
    )
    print(f'Test mAP50-95: {test_metrics.box.map:.4f}')
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    print('Test labels tidak ditemukan; menjalankan prediksi test tanpa menghitung mAP.')
    best_model.predict(
        source=str(DATA_ROOT / 'test' / 'images'),
        imgsz=IMGSZ,
        device=DEVICE,
        conf=0.25,
        save=True,
        save_txt=True,
        project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_test_predictions',
        exist_ok=True,
        verbose=True,
    )
    TEST_OUTPUT_DIR = RUNS_DIR / f'{EXPERIMENT_NAME}_test_predictions'

print(f'Test output: {TEST_OUTPUT_DIR}')


In [ ]:
# 8. Simpan hasil ke satu ZIP yang siap diunduh dari panel Output Kaggle
ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(
    json.dumps(
        {
            'dataset_root': str(DATA_ROOT),
            'model_yaml': str(MODEL_YAML),
            'epochs': EPOCHS,
            'imgsz': IMGSZ,
            'batch': BATCH,
            'optimizer': OPTIMIZER,
            'lr0': LR0,
            'momentum': MOMENTUM,
            'weight_decay': WEIGHT_DECAY,
            'patience': PATIENCE,
            'device': str(DEVICE),
            'seed': SEED,
            'best_checkpoint': str(BEST_PT),
            'last_checkpoint': str(LAST_PT),
        },
        indent=2,
    ),
    encoding='utf-8',
)

def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    """Add a file or directory to the archive and return the number of files added."""
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

log_section('CREATE RESULTS ZIP')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    file_count = 0
    for artifact in (RUN_DIR, TEST_OUTPUT_DIR, DATA_YAML, MODEL_YAML, ECA_MODULE, RUN_CONFIG):
        file_count += add_to_zip(archive, Path(artifact))

print(f'ZIP created : {ZIP_PATH}')
print(f'ZIP size    : {ZIP_PATH.stat().st_size / (1024 ** 2):.2f} MB')
print(f'Files added : {file_count}')
print('Download file tersebut dari panel Output / Files di sisi kanan Kaggle.')

from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
